# Qwen3-0.6B LoRA: expanded Turkish dialogue experiment

This notebook trains a LoRA adapter for the same Qwen3-0.6B base model used by the fast local Q4_K_M GGUF. Training uses the 2,480-conversation Turkish Daily Dialogues 5K train-only derivative (CC BY 4.0). These synthetic conversations are **draft and not individually human-reviewed**. This experiment was explicitly selected for a pilot; it may not improve response quality. The validation and held-out sets are grouped by scenario family.

The notebook downloads the code and licensed training bundle directly from the public GitHub repository; no Kaggle Dataset upload or file selection is needed. Open this notebook in Kaggle, enable a GPU accelerator and internet access, and run all cells. The source attribution and license are included in the bundle. Do not publish the trained model adapter without reviewing the source terms and output policy.

In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path, PurePosixPath

GITHUB_REPO_URL = 'https://github.com/dozagoat47-star/qwen3-turkish-lora-kaggle.git'
GITHUB_REF = 'qwen3-expanded-draft-001'
repo_root = Path('/kaggle/working/qwen3_lora_bundle')
if repo_root.exists():
    shutil.rmtree(repo_root)
subprocess.check_call([
    'git', 'clone', '--depth', '1', '--branch', GITHUB_REF,
    GITHUB_REPO_URL, str(repo_root),
])
bundle_path = repo_root / 'datasets/qwen3_0_6b_lora_bundle_002.zip'
with zipfile.ZipFile(bundle_path) as archive:
    members = archive.infolist()
    for member in members:
        name = PurePosixPath(member.filename)
        if name.is_absolute() or '..' in name.parts:
            raise ValueError(f'Unsafe path in Kaggle bundle: {member.filename}')

    archive.extractall(repo_root)

bundle_manifest = json.loads((repo_root / 'bundle_manifest.json').read_text(encoding='utf-8'))
for relative_path, expected_hash in bundle_manifest['files'].items():
    actual_hash = hashlib.sha256((repo_root / relative_path).read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise ValueError(f'Bundle hash mismatch: {relative_path}')
if bundle_manifest['review_status'] != 'draft' or bundle_manifest['license'] != 'CC BY 4.0':
    raise ValueError('Unexpected dataset review status or license')

data_path = repo_root / 'data/raw/turkish_daily_dialogues_sft_001/train.jsonl'
sys.path.insert(0, str(repo_root))
from clean_model.data import load_jsonl

records = load_jsonl(data_path, require_approved=False)
if len(records) != bundle_manifest['record_count']:
    raise ValueError('Training record count does not match the verified bundle')
if any(record['review']['status'] != 'draft' for record in records):
    raise ValueError('Expected the explicitly opted-in draft dialogue corpus')
if bundle_manifest['source_selection']['source_split'] != 'train only':
    raise ValueError('The bundle must contain only upstream train data')
print(f"Verified {len(records)} draft train conversations; license: {bundle_manifest['license']}")
print(bundle_manifest['attribution'])

subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'transformers>=5,<6', 'peft>=0.17,<1', 'accelerate>=1,<2',
    'huggingface-hub>=1.16,<2',
])
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before training')
print('GPU:', torch.cuda.get_device_name(0))

## Train and select by validation loss

The trainer requires an explicit draft-data opt-in. It keeps whole scenario families in one split, uses validation loss to select the adapter, and does not inspect the held-out split during training. The output directory is intentionally new for this run.

In [ ]:
from clean_model.qwen_lora import train_adapter

run_dir = Path('/kaggle/working/qwen3_0_6b_lora_run_002')
training_metadata = train_adapter(
    dataset_path=data_path,
    output_dir=run_dir,
    epochs=4,
    batch_size=4,
    learning_rate=1e-4,
    patience=2,
    seed=7,
    require_approved=False,
)
print(json.dumps(training_metadata, ensure_ascii=False, indent=2))

## One-shot held-out comparison

Run this only once after training/settings are fixed. It compares the base and LoRA adapter on scenario families withheld from training and validation. Do not use this small qualitative report to repeatedly tune the run.

In [ ]:
import gc

evaluation_lock = run_dir / 'test_evaluation_started.lock'
with evaluation_lock.open('x', encoding='utf-8') as lock:
    lock.write('One-shot held-out evaluation started. Do not rerun for tuning.\n')
gc.collect()
torch.cuda.empty_cache()
from clean_model.qwen_lora import evaluate_test_once

test_report = evaluate_test_once(
    dataset_path=data_path,
    adapter_dir=run_dir,
    report_path=run_dir / 'one_shot_test_evaluation.json',
    seed=7,
    require_approved=False,
)
print(json.dumps(test_report, ensure_ascii=False, indent=2))

## Export a llama.cpp adapter

The LoRA weights are separate from the Q4_K_M base GGUF. This cell converts the trained adapter to GGUF with the same llama.cpp source revision as the local runtime so that the fast llama.cpp server can load both files. Keep the output directory together when downloading the Kaggle results.

In [ ]:
from clean_model.qwen_lora import MODEL_ID, MODEL_REVISION
from transformers import AutoConfig, AutoTokenizer

llama_cpp_commit = 'f9a95aeb2'
llama_cpp_dir = Path('/kaggle/working/llama.cpp')
subprocess.check_call([
    'git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse',
    'https://github.com/ggml-org/llama.cpp.git', str(llama_cpp_dir),
])
subprocess.check_call([
    'git', '-C', str(llama_cpp_dir), 'fetch', '--depth', '1',
    'origin', llama_cpp_commit,
])
subprocess.check_call([
    'git', '-C', str(llama_cpp_dir), 'checkout', '--detach', 'FETCH_HEAD',
])
subprocess.check_call([
    'git', '-C', str(llama_cpp_dir), 'sparse-checkout', 'set', '--no-cone',
    '/convert_lora_to_gguf.py', '/conversion/', '/gguf-py/',
])

base_config_dir = run_dir / 'base_model_config'
base_config_dir.mkdir(exist_ok=False)
AutoConfig.from_pretrained(MODEL_ID, revision=MODEL_REVISION).save_pretrained(base_config_dir)
AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, use_fast=True,
).save_pretrained(base_config_dir)
adapter_gguf_path = run_dir / 'adapter_model.gguf'
subprocess.check_call([
    sys.executable,
    str(llama_cpp_dir / 'convert_lora_to_gguf.py'),
    '--outfile', str(adapter_gguf_path),
    '--outtype', 'f16',
    '--base', str(base_config_dir),
    str(run_dir),
])
adapter_gguf_sha256 = hashlib.sha256(adapter_gguf_path.read_bytes()).hexdigest()
training_metadata['gguf_adapter'] = {
    'file': adapter_gguf_path.name,
    'sha256': adapter_gguf_sha256,
    'format': 'f16 LoRA GGUF',
    'llama_cpp_commit': llama_cpp_commit,
    'base_model_revision': MODEL_REVISION,
}
(run_dir / 'training_metadata.json').write_text(
    json.dumps(training_metadata, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print('GGUF adapter:', adapter_gguf_path, 'bytes:', adapter_gguf_path.stat().st_size)
print('SHA-256:', adapter_gguf_sha256)
import zipfile

deliverables = [
    'adapter_model.gguf', 'adapter_model.safetensors', 'adapter_config.json',
    'training_metadata.json', 'one_shot_test_evaluation.json',
    'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json',
]
output_archive = Path('/kaggle/working/qwen3_0_6b_lora_run_002.zip')
with zipfile.ZipFile(output_archive, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for filename in deliverables:
        path = run_dir / filename
        if path.is_file():
            archive.write(path, arcname=filename)
print('Ready-to-download model bundle:', output_archive)
print('Use its adapter_model.gguf with the existing local Q4_K_M base GGUF.')